# 10 — Input pipeline bottlenecks: prefetch depth sweep

**Learning goal.** Diagnose the most common "TPU at 30% utilisation" failure: the host can't feed the device. Use the prefetch-depth sweep to find where the wait curve flattens.

**What you'll observe.**
- Effective TPU input wait as a function of prefetch depth, for several `device_step_time_s` values.
- The "knee" past which more prefetch buys nothing.

**Knobs to tune.**
- `device_step_time_s` — how fast the TPU consumes batches. Faster TPUs are harder to feed.
- `batch_size`, `bytes_per_sample` — how big each batch is.
- `disk_bandwidth_bytes_s`, `preprocess_s_per_sample`, `h2d_bandwidth_bytes_s` inside `simulate_input_pipeline`.

**Failure modes to watch.**
- If wait stays high regardless of depth, the host pipeline itself is too slow per batch — prefetch can't help.
- Prefetch depth costs host RAM. Don't set it absurdly high "to be safe".

In [ ]:
import sys, os
from pathlib import Path
# Add the parent of cloud_tpu_lab to sys.path so `cloud_tpu_lab.src.*` imports work.
HERE = Path(os.getcwd()).resolve()
_root = HERE
for _ in range(5):
    if (_root / "cloud_tpu_lab").exists():
        break
    _root = _root.parent
sys.path.insert(0, str(_root))
sys.path.insert(0, "..")
PLOT_DIR = Path("../artifacts/plots")
PLOT_DIR.mkdir(parents=True, exist_ok=True)
print("repo root:", _root)
print("plot dir:", PLOT_DIR.resolve())

## Sweep prefetch depth for several device step times

In [ ]:
from cloud_tpu_lab.src.input_pipeline.prefetch_sim import sweep_prefetch_depth

depths = [0, 1, 2, 4, 8, 16, 32]
device_step_times = [0.001, 0.005, 0.020, 0.080]   # fast TPU vs slow TPU

results = {}
for dst in device_step_times:
    sweep = sweep_prefetch_depth(
        depths=depths,
        batch_size=32,
        bytes_per_sample=4 * 224 * 224 * 3,   # ~600 KB / sample (image-ish)
        device_step_time_s=dst,
    )
    results[dst] = sweep
    print(f"\ndevice_step={dst*1000:.1f} ms")
    for d, w in sweep:
        print(f"  prefetch={d:>2}  wait={w*1000:7.3f} ms")

## Plot: wait vs prefetch depth

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(8, 5))
for dst, sweep in results.items():
    xs = [d for d, _ in sweep]
    ys = [w * 1000 for _, w in sweep]
    plt.plot(xs, ys, marker="o", label=f"device_step={dst*1000:.0f} ms")

plt.xlabel("prefetch depth (batches staged ahead)")
plt.ylabel("effective TPU input wait (ms)")
plt.title("Input wait vs prefetch depth")
plt.legend(title="TPU compute / step")
plt.grid(True, alpha=0.3)
plt.tight_layout()
out_png = PLOT_DIR / "nb10_prefetch_sweep.png"
plt.savefig(out_png, dpi=120)
plt.show()
print("saved:", out_png)

## How to read the plot

- The **fast-TPU** lines (device_step small) stay high because the host pipeline can't keep up no matter how deep the prefetch buffer is. The fix isn't prefetch — it's a faster pipeline (more workers, tf.data parallel `map`, larger batch, smaller bytes_per_sample).
- The **slow-TPU** lines hit zero quickly: even depth 1 or 2 is enough.
- The "knee" is where adding more depth stops helping. Pick prefetch a touch past the knee.

## Exercises

1. Drop `bytes_per_sample` to 4 KB (a tiny embedding lookup). Does the slowest TPU still see any wait?
2. Triple `preprocess_s_per_sample` inside a direct call to `simulate_input_pipeline`. Plot how it shifts the curves.
3. The simulator uses a single pipeline-stage model. In real `tf.data`, parallel `map(num_parallel_calls=AUTOTUNE)` partly absorbs preprocess. Model that by dividing `preprocess_s_per_sample` by N_workers and replot.
4. At what `device_step_time_s` does prefetch=2 suffice to fully overlap the pipeline for this default config?
5. Why is prefetch beneficial only up to a point? Express it as an inequality in `pipeline_s` and `device_step_time_s * depth`.